# Ejercicio 7: Bases de Datos Vectoriales

## Objetivo de la práctica

Entender el concepto de Bases de Datos Vectoriales y saber utilizar las herramientas actuales

## Parte 0: Carga del Corpus

Vamos a utilizar la API de Kaggle para acceder al dataset _Wikipedia Text Corpus for NLP and LLM Projects_

El corpus está disponible desde este [link](https://www.kaggle.com/datasets/gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects?utm_source=chatgpt.com)

### Actividad

1. Carga el corpus


In [1]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

In [2]:
# Set the path to the file you'd like to load
file_path = "wikipedia_text_corpus.csv"

# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects",
  file_path,
)

df.head()

,Unnamed: 0,text
0,1,Anovo\n\nAnovo (formerly A Novo) is a computer...
1,2,Battery indicator\n\nA battery indicator (also...
2,3,"Bob Pease\n\nRobert Allen Pease (August 22, 19..."
3,4,CAVNET\n\nCAVNET was a secure military forum w...
4,5,CLidar\n\nThe CLidar is a scientific instrumen...


## Parte 1: Generación de Embeddings

Vamos a utilizar E5 como modelo de embeddings.

La documentación de E5 está disponible desde este [link](https://huggingface.co/intfloat/e5-base-v2)

### Actividad

1. Normalizar el corpus
2. Definir una función `chunk_text`, y dividir los textos en _chunks_.
3. Generar embeddings por cada _chunk_

In [3]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import re

df = df.dropna(subset=["text"]).reset_index(drop=True)

# Limpieza básica
def normalize_text(s: str) -> str:
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["text_norm"] = df["text"].astype(str).map(normalize_text)

df.head()

,Unnamed: 0,text,text_norm
0,1,Anovo\n\nAnovo (formerly A Novo) is a computer...,Anovo Anovo (formerly A Novo) is a computer se...
1,2,Battery indicator\n\nA battery indicator (also...,Battery indicator A battery indicator (also kn...
2,3,"Bob Pease\n\nRobert Allen Pease (August 22, 19...","Bob Pease Robert Allen Pease (August 22, 1940Â..."
3,4,CAVNET\n\nCAVNET was a secure military forum w...,CAVNET CAVNET was a secure military forum whic...
4,5,CLidar\n\nThe CLidar is a scientific instrumen...,CLidar The CLidar is a scientific instrument u...


In [4]:
def chunk_text(text: str, max_chars: int = 800, overlap: int = 100):
    """
    Chunking por caracteres.
    max_chars ~ 600-1000 suele funcionar bien.
    overlap ayuda a no cortar ideas a la mitad.
    """
    chunks = []
    start = 0
    n = len(text)
    while start < n:
        end = min(start + max_chars, n)
        chunk = text[start:end]
        chunk = chunk.strip()
        if len(chunk) > 0:
            chunks.append(chunk)
        if end == n:
            break
        start = max(0, end - overlap)
    return chunks

records = []
for i, row in df.iterrows():
    chunks = chunk_text(row["text_norm"], max_chars=800, overlap=100)
    for j, ch in enumerate(chunks):
        records.append({
            "doc_id": int(i),
            "chunk_id": j,
            "text": ch
        })

chunks_df = pd.DataFrame(records)
chunks_df.head(), len(chunks_df)

(   doc_id  chunk_id                                               text
 0       0         0  Anovo Anovo (formerly A Novo) is a computer se...
 1       1         0  Battery indicator A battery indicator (also kn...
 2       1         1  ad battery when in reality it indicates a prob...
 3       1         2  s that an internal standby battery needs repla...
 4       1         3  increase; in many cases the EMF remains more o...,
 79104)

In [5]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "intfloat/e5-base-v2"   # recomendado para retrieval
model = SentenceTransformer(MODEL_NAME)

# Textos a indexar (pasajes)
passages = ["passage: " + t for t in chunks_df["text"].tolist()]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/e5-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
# Pool para que use las 2 GPUs de Kaggle en paralelo
pool = model.start_multi_process_pool()
# Embeddings (N x D)
# Se debe usar normalize_embeddings=True para similitud coseno
embeddings = model.encode(
    passages,
    pool=pool, # Le pasamos el pool
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

model.stop_multi_process_pool(pool)

Chunks:   0%|          | 0/20 [00:00<?, ?it/s]

In [7]:
print(embeddings.shape, embeddings.dtype)

(79104, 768) float32


In [8]:
def embed_query(query: str) -> np.ndarray:
    q = "query: " + query
    vec = model.encode(
        [q],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")
    return vec

query_text = "Battery measuring"

query_vec = embed_query(query_text)
query_vec.shape

(1, 768)

## Parte 2: FAISS

FAISS es una librería para búsqueda por similitud eficiente y clustering de vectores densos.

La documentación de FAISS está disponible en este [link](https://faiss.ai/index.html)

### Actividad

1. Crea un índice en FAISS
2. Carga los embeddings
3. Realiza una búsqueda a partir de una _query_

In [39]:
!pip install -q faiss-cpu
# código base para FAISS
import faiss
import numpy as np

# Asumiendo `embeddings` en un array NxD
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

D, I = index.search(query_vec, k=10)

print("Distancias (top-10):", D)
print("IDs (top-10):", I)

I0702 20:30:20.998111     448 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


Distancias (top-10): [[0.25930262 0.2763996  0.31979656 0.3217336  0.32282174 0.33097768
  0.33130074 0.33675623 0.33956766 0.3467201 ]]
IDs (top-10): [[10176     1 10177 37406 71872 37409 10481     5 75249 47064]]


## Parte 3 — Vector DB #1: Qdrant (búsqueda vectorial + metadata)

### Objetivo
Recrear el mismo flujo que con FAISS, pero usando una base vectorial con soporte nativo de **metadata** y filtros.

### Qué debes implementar
1. Levantar / conectar con una instancia de Qdrant.
2. Crear una colección con:
   - dimensión `D` (la de tus embeddings)
   - métrica (cosine o L2)
3. Insertar:
   - `id`
   - `embedding`
   - `payload` (metadata: texto, título, etiquetas, etc.)
4. Consultar Top-k por similitud:
   - `query_embedding`
   - `k`

### Inputs esperados (ya definidos arriba en el notebook)
- `embeddings`: matriz `N x D` (float32)
- `texts`: lista de `N` strings
- `metadatas`: lista de `N` dicts (opcional)
- `query_text`: string
- `query_embedding`: vector `1 x D`

### Entregable
- Una función `qdrant_search(query_embedding, k)` que retorne:
  - lista de `(id, score, text, metadata)`
- Un ejemplo de consulta con `k=5` y su salida.

### Preguntas
- ¿La métrica usada fue cosine o L2? ¿Por qué?
- ¿Qué tan fácil fue filtrar por metadata en comparación con FAISS?
- ¿Qué pasa con el tiempo de respuesta cuando aumentas `k`?


In [10]:
# Instalar el cliente de Qdrant
!pip install -q qdrant-client

from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

# 1. Levantar instancia de Qdrant en memoria
client = QdrantClient(":memory:")
collection_name = "wikipedia_collection"

# La dimensión de los embeddings generados por E5 en la Parte 1
dim = embeddings.shape[1] 

# 2. Crear la colección con métrica Cosine
client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=dim, distance=Distance.COSINE),
)

# 3. Insertar id, embedding y payload (metadata)
points = []
for idx, row in chunks_df.iterrows():
    points.append(
        PointStruct(
            id=idx,
            vector=embeddings[idx].tolist(),
            payload={
                "doc_id": row["doc_id"],
                "chunk_id": row["chunk_id"],
                "text": row["text"]
            }
        )
    )

# Subimos los vectores a la colección
client.upsert(
    collection_name=collection_name,
    points=points
)

# 4. Función de búsqueda Top-k (Actualizada para qdrant-client v1.16+)
def qdrant_search(query_embedding, k=5):
    # Aseguramos que el vector sea una lista plana
    query_list = query_embedding[0].tolist() if query_embedding.ndim == 2 else query_embedding.tolist()
    
    # Usamos query_points en lugar del antiguo search, y el parámetro de entrada es 'query'
    response = client.query_points(
        collection_name=collection_name,
        query=query_list,
        limit=k
    )
    
    # Formatear la salida: (id, score, text, metadata)
    results = []
    
    # Los resultados ahora viven dentro del atributo .points de la respuesta
    for hit in response.points:
        metadata = {
            "doc_id": hit.payload.get("doc_id"), 
            "chunk_id": hit.payload.get("chunk_id")
        }
        text = hit.payload.get("text")
        results.append((hit.id, hit.score, text, metadata))
        
    return results

# Prueba de la función
print("=== Resultados Qdrant (k=5) ===")
resultados = qdrant_search(query_vec, k=5)
for res in resultados:
    print(f"ID: {res[0]} | Score: {res[1]:.4f}")
    print(f"Metadata: {res[3]}")
    print(f"Texto: {res[2][:150]}...\n")

/tmp/ipykernel_448/174964708.py:36: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Current collection contains 79104 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  client.upsert(


=== Resultados Qdrant (k=5) ===
ID: 10176 | Score: 0.8703
Metadata: {'doc_id': 1391, 'chunk_id': 0}
Texto: Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going from a simple device for testing ...

ID: 1 | Score: 0.8618
Metadata: {'doc_id': 1, 'chunk_id': 0}
Texto: Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a battery. This will usually be a visu...

ID: 10177 | Score: 0.8401
Metadata: {'doc_id': 1391, 'chunk_id': 1}
Texto: ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries. Their common principle is ba...

ID: 37406 | Score: 0.8391
Metadata: {'doc_id': 5067, 'chunk_id': 1}
Texto: ils. One was connected via a series resistor to the battery supply. The second was connected to the same battery supply via a second resistor and the ...

ID: 71872 | Score: 0.8386
Metadata: {'doc_id': 9888, 'c

**¿La métrica usada fue cosine o L2? ¿Por qué?**  
Se usó Distance.COSINE. Es la métrica recomendada cuando se trabaja con embeddings de modelos tipo E5, porque estos modelos están entrenados/optimizados para que la similitud semántica se mida por el ángulo entre vectores, no por su magnitud. Además, en tu pipeline ya normalizaste los embeddings (normalize_embeddings=True), lo cual es coherente con usar cosine.  
**¿Qué tan fácil fue filtrar por metadata en comparación con FAISS?**  
Con FAISS no hay soporte nativo para metadata ni filtros: el índice solo almacena vectores, y cualquier filtro hay que hacerlo "a mano" fuera del índice, buscando primero y filtrando después, o manteniendo estructuras paralelas. Con Qdrant, el payload se guarda junto al vector y se pueden aplicar filtros directamente en la consulta (con el parámetro query_filter), lo cual es mucho más directo y eficiente.  
**¿Qué pasa con el tiempo de respuesta cuando aumentas k?**  
En general el tiempo de consulta crece, pero no de forma dramática para valores pequeños/medianos de k, ya que Qdrant usa un índice HNSW aproximado, la mayor parte del costo está en atravesar el grafo, y extraer más candidatos del top-k añade un costo marginal relativamente bajo.

## Parte 4 — Vector DB #2: Milvus (indexación ANN y escalabilidad)

### Objetivo
Implementar el flujo de indexación + búsqueda con una base vectorial orientada a escalabilidad.

### Qué debes implementar
1. Conectar a Milvus.
2. Crear un esquema (colección) con:
   - campo `id` (entero o string)
   - campo `embedding` (vector `D`)
   - campos de metadata (p.ej., `category`, `source`, `title`)
3. Insertar `N` embeddings.
4. Crear/seleccionar un índice ANN (ej. HNSW o IVF).
5. Ejecutar consultas Top-k y recuperar textos asociados.

### Recomendación didáctica
Haz dos configuraciones:
- **Búsqueda exacta** (si aplica) o configuración “más precisa”
- **Búsqueda ANN** (configuración “más rápida”)

Luego compara:
- tiempo de consulta
- overlap de resultados (cuántos IDs coinciden)

### Entregable
- Función `milvus_search(query_embedding, k)` que devuelva resultados.
- Un mini experimento: `k=5` y `k=20` (tiempos y resultados).

### Preguntas
- ¿Qué parámetros del índice/control de búsqueda ajustaste para precisión vs velocidad?
- ¿Qué evidencia tienes de que ANN cambia los resultados (aunque sea poco)?


In [16]:
# Instalar pymilvus
!pip install -q "pymilvus[milvus_lite]"

from pymilvus import MilvusClient, DataType
import time

# 1. Conectar a Milvus
client = MilvusClient("milvus_kaggle.db")
collection_name = "wiki_milvus"

# Limpiamos si ya existía para evitar errores al re-ejecutar
if client.has_collection(collection_name):
    client.drop_collection(collection_name)

# 2. Crear el esquema
dim = embeddings.shape[1]

schema = client.create_schema(auto_id=False, enable_dynamic_field=True)
schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True)
schema.add_field(field_name="vector", datatype=DataType.FLOAT_VECTOR, dim=dim)

client.create_collection(
    collection_name=collection_name,
    schema=schema
)

# 3. Insertar N embeddings
print("Preparando datos...")
data_to_insert = []
for idx, row in chunks_df.iterrows():
    data_to_insert.append({
        "id": int(idx),
        "vector": embeddings[idx].tolist(),
        "text": row["text"],
        "doc_id": int(row["doc_id"])
    })

batch_size = 10000
print(f"Insertando datos en Milvus en lotes de {batch_size}...")

for i in range(0, len(data_to_insert), batch_size):
    batch = data_to_insert[i : i + batch_size]
    client.insert(
        collection_name=collection_name,
        data=batch
    )
    print(f"Procesado lote: {i} a {min(i + batch_size, len(data_to_insert))}")

print("¡Inserción completada con éxito!")

# 4. Crear índice ANN
print("Configurando índice...")

index_params = client.prepare_index_params()
index_params.add_index(
    field_name="vector",
    index_type="HNSW",
    metric_type="COSINE",
    params={"M": 16, "efConstruction": 200}
)

# Esto ahora correrá a la primera sin errores
client.create_index(
    collection_name=collection_name,
    index_params=index_params
)
print("Índice HNSW creado exitosamente.")

# Cargamos la colección para poder hacer búsquedas
client.load_collection(collection_name)
print("Colección cargada en memoria y lista para consultas.")

# 5. Función de búsqueda Top-k
def milvus_search(query_embedding, k=5, search_params=None):
    query_list = query_embedding[0].tolist() if query_embedding.ndim == 2 else query_embedding.tolist()

    start_time = time.time()
    results = client.search(
        collection_name=collection_name,
        data=[query_list],
        limit=k,
        search_params=search_params,
        output_fields=["text", "doc_id"]
    )
    end_time = time.time()
    elapsed = end_time - start_time

    # Formatear resultados: (id, score, text, metadata)
    formatted = []
    for hit in results[0]:
        entity = hit.get("entity", {})
        metadata = {"doc_id": entity.get("doc_id")}
        text = entity.get("text")
        formatted.append((hit["id"], hit["distance"], text, metadata))

    return formatted, elapsed

# --- Experimento (k=5 vs k=20 y Exacta vs ANN) ---
ann_params = {"metric_type": "COSINE", "params": {"ef": 10}}
exact_params = {"metric_type": "COSINE", "params": {"ef": 200}}

# Warm-up: no medimos esta llamada, solo "calienta" la conexión/índice
_ = milvus_search(query_vec, k=5, search_params=ann_params)

# A. Búsqueda Rápida ANN (ef bajo)
res_ann_5, t_ann_5 = milvus_search(query_vec, k=5, search_params=ann_params)

# B. Búsqueda "Exhaustiva" o más precisa (ef alto)
res_exact_5, t_exact_5 = milvus_search(query_vec, k=5, search_params=exact_params)

# C. ANN con k=20
res_ann_20, t_ann_20 = milvus_search(query_vec, k=20, search_params=ann_params)

print(f"\n--- Resultados Experimento ---")
print(f"Tiempo ANN (k=5, ef=10): {t_ann_5:.4f} seg")
print(f"Tiempo Preciso (k=5, ef=200): {t_exact_5:.4f} seg")
print(f"Tiempo ANN (k=20, ef=10): {t_ann_20:.4f} seg")

print("\nTop 3 IDs (Búsqueda Precisa vs Búsqueda ANN):")
ids_exact = [hit[0] for hit in res_exact_5[:3]]
ids_ann = [hit[0] for hit in res_ann_5[:3]]
print(f"IDs Preciso: {ids_exact}")
print(f"IDs ANN:     {ids_ann}")

overlap = set(ids_exact).intersection(set(ids_ann))
print(f"Overlap: {len(overlap)}/3 -> {overlap}")

I0702 20:01:09.329897     448 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


Preparando datos...
Insertando datos en Milvus en lotes de 10000...
Procesado lote: 0 a 10000
Procesado lote: 10000 a 20000
Procesado lote: 20000 a 30000
Procesado lote: 30000 a 40000
Procesado lote: 40000 a 50000
Procesado lote: 50000 a 60000
Procesado lote: 60000 a 70000
Procesado lote: 70000 a 79104
¡Inserción completada con éxito!
Configurando índice...
Índice HNSW creado exitosamente.
Colección cargada en memoria y lista para consultas.

--- Resultados Experimento ---
Tiempo ANN (k=5, ef=10): 3.5617 seg
Tiempo Preciso (k=5, ef=200): 3.5698 seg
Tiempo ANN (k=20, ef=10): 3.5424 seg

Top 3 IDs (Búsqueda Precisa vs Búsqueda ANN):
IDs Preciso: [10176, 1, 10177]
IDs ANN:     [10176, 1, 10177]
Overlap: 3/3 -> {10176, 1, 10177}


**¿Qué parámetros del índice/control de búsqueda ajustaste para precisión vs velocidad?**  
Se ajustó el parámetro `ef` de búsqueda en HNSW: `ef=10` prioriza velocidad (explora
menos nodos del grafo) y `ef=200` prioriza precisión (explora más nodos, acercándose
más al resultado exacto). En los tiempos medidos, la diferencia fue pequeña, sugiriendo que en este dataset el tiempo total está dominado por el overhead de la llamada más que por el propio cómputo del índice.
Al aumentar k de 5 a 20 con ef=10, el tiempo se mantuvo prácticamente igual, lo cual es coherente con cómo funciona HNSW: el costo principal es
recorrer el grafo, y extraer más resultados del top-k es relativamente barato.

**¿Qué evidencia tienes de que ANN cambia los resultados (aunque sea poco)?**  
Para la query usada, los IDs top-3 obtenidos con ef=10 (ANN "rápido") y ef=200
(más preciso) fueron idénticos: {10176, 1, 10177}, dando un overlap de 3/3.
Esto indica que, para esta consulta puntual, los documentos más relevantes están
lo suficientemente bien separados del resto en el espacio de embeddings como para
que incluso una búsqueda aproximada con ef bajo los encuentre sin error. No se
observó degradación en este caso, aunque en datasets más grandes o consultas más
"ambiguas" sí podría esperarse mayor divergencia entre ANN y búsqueda exacta.

## Parte 5 — Vector DB #3: Weaviate (búsqueda semántica con esquema)

### Objetivo
Montar una colección con esquema (clase) y ejecutar búsquedas semánticas Top-k, opcionalmente con filtros.

### Qué debes implementar
1. Conectar a Weaviate.
2. Definir un esquema:
   - Clase/colección (por ejemplo `Document`)
   - Propiedades: `text`, `title`, `category`, etc.
   - Vector asociado (embedding)
3. Insertar objetos con:
   - propiedades + vector
4. Consultar por similitud (Top-k) con `query_embedding`.
5. (Opcional) agregar un filtro por propiedad (metadata).

### Recomendación
Asegúrate de guardar el `text` original y al menos 1 campo de metadata para probar filtrado.

### Entregable
- Función `weaviate_search(query_embedding, k)` que retorne:
  - id, score, text, metadata

### Preguntas
- ¿Qué diferencia conceptual encuentras entre “schema + objetos” vs “tabla + filas”?
- ¿Cómo describirías el trade-off de complejidad vs expresividad?


In [17]:
!pip install -q weaviate-client
import weaviate
import weaviate.classes as wvc
import os
os.environ["LOG_LEVEL"] = "error"

# 1. Conectar a Weaviate
client = weaviate.connect_to_embedded()

if client.is_connected():
    print("¡Conectado exitosamente a Weaviate!")
else:
    print("No se pudo conectar.")

# 2. Definición del Esquema (Schema)
collection_name = "WikipediaArticle"

if client.collections.exists(collection_name):
    client.collections.delete(collection_name)

collection = client.collections.create(
    name=collection_name,
    vectorizer_config=wvc.config.Configure.Vectorizer.none(),
    properties=[
        wvc.config.Property(name="text", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="doc_id", data_type=wvc.config.DataType.INT),
    ]
)

print(f"Colección '{collection_name}' creada.")

# 3. Insertar datos en Weaviate
print("Insertando objetos...")

objects_to_insert = [
    {
        "text": chunks_df.iloc[i]["text"],
        "doc_id": int(chunks_df.iloc[i]["doc_id"])
    }
    for i in range(len(chunks_df))
]

with collection.batch.dynamic() as batch:
    for i, obj in enumerate(objects_to_insert):
        batch.add_object(
            properties=obj,
            vector=embeddings[i].tolist()
        )

print(f"¡Se insertaron {len(objects_to_insert)} objetos en Weaviate!")

if collection.batch.failed_objects:
    print(f"Objetos con error: {len(collection.batch.failed_objects)}")
    print(collection.batch.failed_objects[:3])

total = collection.aggregate.over_all(total_count=True)
print(f"Objetos realmente insertados: {total.total_count}")

# 4. Función de búsqueda Top-k
def weaviate_search(query_embedding, k=5, doc_id_filter=None):
    query_list = query_embedding[0].tolist() if query_embedding.ndim == 2 else query_embedding.tolist()

    filters = None
    if doc_id_filter is not None:
        filters = wvc.query.Filter.by_property("doc_id").equal(doc_id_filter)

    response = collection.query.near_vector(
        near_vector=query_list,
        limit=k,
        filters=filters,
        return_metadata=wvc.query.MetadataQuery(distance=True)
    )

    results = []
    for obj in response.objects:
        metadata = {"doc_id": obj.properties.get("doc_id")}
        text = obj.properties.get("text")
        score = obj.metadata.distance
        results.append((obj.uuid, score, text, metadata))

    return results

# 5. Prueba sin filtro
print("\n=== Resultados Weaviate (k=5, sin filtro) ===")
resultados = weaviate_search(query_vec, k=5)
for res in resultados:
    print(f"ID: {res[0]} | Distancia: {res[1]:.4f}")
    print(f"Metadata: {res[3]}")
    print(f"Texto: {res[2][:150]}...\n")

# 6. Prueba con filtro por doc_id
doc_id_ejemplo = resultados[0][3]["doc_id"]
print(f"=== Resultados Weaviate (k=5, filtrado por doc_id={doc_id_ejemplo}) ===")
resultados_filtrados = weaviate_search(query_vec, k=5, doc_id_filter=doc_id_ejemplo)
for res in resultados_filtrados:
    print(f"ID: {res[0]} | Distancia: {res[1]:.4f}")
    print(f"Metadata: {res[3]}")
    print(f"Texto: {res[2][:150]}...\n")

# 7. Cerrar conexión
client.close()

I0702 20:04:16.943407     448 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
grpcio-tools 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.
¡Conectado exitosamente a Weaviate!


{"action":"load_all_shards","build_git_commit":"","build_go_version":"go1.24.3","build_image_tag":"","build_wv_version":"1.30.5","level":"error","msg":"failed to load all shards: context canceled","time":"2026-07-02T20:04:25Z"}
/usr/local/lib/python3.12/dist-packages/weaviate/warnings.py:196: DeprecationWarning: Dep024: You are using the `vectorizer_config` argument in `collection.config.create()`, which is deprecated.
            Use the `vector_config` argument instead.
            
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Colección 'WikipediaArticle' creada.
Insertando objetos...


I0702 20:05:10.477798     722 chttp2_transport.cc:1376] ipv4:127.0.0.1:36525: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {http2_error:11}
E0702 20:05:10.477855     722 chttp2_transport.cc:1408] ipv4:127.0.0.1:36525: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 10000ms


¡Se insertaron 79104 objetos en Weaviate!
Objetos realmente insertados: 79104

=== Resultados Weaviate (k=5, sin filtro) ===
ID: aa5581e7-23ae-4174-a69e-dd6aca665c98 | Distancia: 0.1297
Metadata: {'doc_id': 1391}
Texto: Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going from a simple device for testing ...

ID: 4fcfb917-d9ee-40d8-b8bc-2528481acf53 | Distancia: 0.1382
Metadata: {'doc_id': 1}
Texto: Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a battery. This will usually be a visu...

ID: d7a8fd3b-ebbb-49b8-b4d6-d1a8b0674393 | Distancia: 0.1599
Metadata: {'doc_id': 1391}
Texto: ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries. Their common principle is ba...

ID: b8de6c42-2543-40d7-aebb-9ed3833d310c | Distancia: 0.1609
Metadata: {'doc_id': 5067}
Texto: ils. One was connected via a seri

{"build_git_commit":"","build_go_version":"go1.24.3","build_image_tag":"","build_wv_version":"1.30.5","error":"cannot find peer","level":"error","msg":"transferring leadership","time":"2026-07-02T20:05:52Z"}


**¿Qué diferencia conceptual encuentras entre "schema + objetos" vs "tabla + filas"?**  
En Weaviate se define una colección con un esquema explícito de propiedades tipadas
(en este caso `text` y `doc_id`), y cada registro es un "objeto" con esas propiedades
más un vector asociado de forma nativa. Es conceptualmente similar a una tabla
relacional (propiedades tipadas = columnas, objetos = filas), pero el vector es un
componente de primera clase del objeto: la búsqueda por similitud (`near_vector`) y
el filtrado por propiedades (`Filter.by_property`) se combinan en una sola consulta
nativa, sin necesidad de mezclar SQL con una extensión externa.

**¿Cómo describirías el trade-off de complejidad vs expresividad?**  
Requiere más configuración inicial que herramientas como Chroma (definir esquema,
tipos de propiedades, desactivar el vectorizer automático), y hay que estar atento
a validar la inserción por batch (como se vio, el conteo intermedio de objetos
puede no reflejar el estado final). A cambio, se obtiene una API de consulta más
expresiva: se pudo filtrar exactamente por `doc_id=1391` y obtener solo 2 de los
5 resultados originales, combinando similitud semántica y metadata en una sola
llamada — algo que en FAISS habría requerido lógica adicional manual.

## Parte 6 — Vector Store #4: Chroma (prototipado rápido)

### Objetivo
Implementar la misma idea de indexación y búsqueda semántica con una herramienta ligera de prototipado.

### Qué debes implementar
1. Crear una colección.
2. Insertar:
   - ids
   - embeddings
   - documents (texto)
   - metadatas (opcional)
3. Consultar Top-k con `query_embedding`.

### Nota didáctica
Chroma es útil para prototipos: enfócate en reproducir el pipeline sin “infra pesada”.

### Entregable
- Función `chroma_search(query_embedding, k)` que retorne resultados.
- Una consulta con `k=5`.

### Preguntas
- ¿Qué tan fácil fue implementar todo comparado con Qdrant/Milvus?
- ¿Qué limitaciones ves para un sistema en producción?


In [19]:
!pip install -q chromadb
!pip install -q --force-reinstall "opentelemetry-api==1.38.0" "opentelemetry-sdk==1.38.0" "opentelemetry-exporter-otlp-proto-common==1.38.0" "opentelemetry-proto==1.38.0" "opentelemetry-exporter-otlp-proto-http==1.38.0" "opentelemetry-exporter-otlp-proto-grpc==1.38.0"
import chromadb

# Cliente en memoria (efímero, ideal para prototipado)
chroma_client = chromadb.Client()

collection_name = "wikipedia_chroma"

# Limpiar si ya existía (por si re-ejecutas la celda)
if collection_name in [c.name for c in chroma_client.list_collections()]:
    chroma_client.delete_collection(collection_name)

chroma_collection = chroma_client.create_collection(
    name=collection_name,
    metadata={"hnsw:space": "cosine"}  # métrica coseno, coherente con las otras partes
)

I0702 20:07:57.266549     448 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
/usr/lib/python3.12/pty.py:95: DeprecationWarning: This process (pid=448) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()
I0702 20:08:01.251528     448 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
weaviate-client 4.22.0 requires grpcio<1.80.0,>=1.59.5, but you have grpcio 1.81.1 which is incompatible.
grpcio-tools 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4

In [22]:
ids = [str(idx) for idx in chunks_df.index]
documents = chunks_df["text"].tolist()
metadatas = [{"doc_id": int(row["doc_id"]), "chunk_id": int(row["chunk_id"])} for _, row in chunks_df.iterrows()]
embeddings_list = embeddings.tolist()

batch_size = 5000
print("Insertando en Chroma...")
for i in range(0, len(ids), batch_size):
    chroma_collection.add(
        ids=ids[i:i+batch_size],
        embeddings=embeddings_list[i:i+batch_size],
        documents=documents[i:i+batch_size],
        metadatas=metadatas[i:i+batch_size]
    )
    print(f"Procesado lote: {i} a {min(i+batch_size, len(ids))}")

print(f"¡Se insertaron {chroma_collection.count()} objetos en Chroma!")

Insertando en Chroma...
Procesado lote: 0 a 5000
Procesado lote: 5000 a 10000
Procesado lote: 10000 a 15000
Procesado lote: 15000 a 20000
Procesado lote: 20000 a 25000
Procesado lote: 25000 a 30000
Procesado lote: 30000 a 35000
Procesado lote: 35000 a 40000
Procesado lote: 40000 a 45000
Procesado lote: 45000 a 50000
Procesado lote: 50000 a 55000
Procesado lote: 55000 a 60000
Procesado lote: 60000 a 65000
Procesado lote: 65000 a 70000
Procesado lote: 70000 a 75000
Procesado lote: 75000 a 79104
¡Se insertaron 79104 objetos en Chroma!


In [26]:
def chroma_search(query_embedding, k=5):
    query_list = query_embedding[0].tolist() if query_embedding.ndim == 2 else query_embedding.tolist()

    results = chroma_collection.query(
        query_embeddings=[query_list],
        n_results=k
    )

    formatted = []
    for i in range(len(results["ids"][0])):
        id_ = results["ids"][0][i]
        distance = results["distances"][0][i]
        text = results["documents"][0][i]
        metadata = results["metadatas"][0][i]
        formatted.append((id_, distance, text, metadata))

    return formatted

print("=== Resultados Chroma (k=5) ===")
resultados = chroma_search(query_vec, k=5)
for res in resultados:
    print(f"ID: {res[0]} | Distancia: {res[1]:.4f}")
    print(f"Metadata: {res[3]}")
    print(f"Texto: {res[2][:150]}...\n")

=== Resultados Chroma (k=5) ===
ID: 10176 | Distancia: 0.1297
Metadata: {'doc_id': 1391, 'chunk_id': 0}
Texto: Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going from a simple device for testing ...

ID: 1 | Distancia: 0.1382
Metadata: {'doc_id': 1, 'chunk_id': 0}
Texto: Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a battery. This will usually be a visu...

ID: 10177 | Distancia: 0.1599
Metadata: {'doc_id': 1391, 'chunk_id': 1}
Texto: ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries. Their common principle is ba...

ID: 37406 | Distancia: 0.1609
Metadata: {'chunk_id': 1, 'doc_id': 5067}
Texto: ils. One was connected via a series resistor to the battery supply. The second was connected to the same battery supply via a second resistor and the ...

ID: 71872 | Distancia: 0.1614
Metadata:

**¿Qué tan fácil fue implementar todo comparado con Qdrant/Milvus?**  
Chroma fue la más simple de las cuatro ya que no requiere definir un esquema explícito
como Weaviate, ni configurar métricas/índices ANN manualmente como Milvus. La API de
`add()` y `query()` es directa y minimalista, comparable a Qdrant en simplicidad,
pero sin necesidad de definir `VectorParams` o estructuras `PointStruct`.

**¿Qué limitaciones ves para un sistema en producción?**  
El cliente usado aquí es en memoria/efímero: los datos no persisten entre sesiones a menos que se use un cliente persistente o un servidor Chroma dedicado. Además, en este entorno se encontraron conflictos de dependencias al instalarlo, lo que lo que hace notar que es una herramienta más orientada a prototipado rápido que a despliegues productivos robustos, donde herramientas como Qdrant o Milvus ofrecen más garantías de escalabilidad, persistencia y estabilidad de dependencias.

## Parte 7 — SQL + vectores: PostgreSQL/pgvector (vector search transparente)

### Objetivo
Guardar embeddings en una tabla y ejecutar una consulta SQL de similitud.

### Qué debes implementar
1. Conectar a una base PostgreSQL con `pgvector` habilitado.
2. Crear una tabla (ej. `documents`) con:
   - `id` (PK)
   - `text` (texto)
   - `embedding` (vector(D))
   - metadata (columnas adicionales)
3. Insertar todos los documentos y embeddings.
4. Consultar Top-k por similitud, ordenando por distancia.

### Fórmula conceptual (lo que implementa tu SQL)
Para una consulta `q`, buscas:
$$ argmin_d \in D \; \text{dist}(\vec{q}, \vec{d})$$
donde `dist` puede ser L2 o una variante para cosine (según configuración).

### Entregable
- Función `pgvector_search(query_embedding, k)` que ejecute SQL y devuelva:
  - id, score/distancia, text, metadata

### Preguntas
- ¿Qué tan “explicable” te parece esta aproximación vs las otras?
- ¿Qué ventajas ofrece el mundo SQL (JOIN, filtros, agregaciones)?
- ¿Qué limitaciones esperas en escalabilidad frente a bases vectoriales dedicadas?


In [27]:
# Instalar PostgreSQL y herramientas necesarias
!apt-get update -qq
!apt-get install -qq -y postgresql postgresql-contrib > /dev/null

# Instalar la extensión pgvector (compilar desde source, ya que apt no la trae)
!apt-get install -qq -y postgresql-server-dev-all gcc make git > /dev/null
!git clone --quiet https://github.com/pgvector/pgvector.git
%cd pgvector
!make -s
!make -s install
%cd ..

I0702 20:14:07.422426     448 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


I0702 20:14:13.204918     448 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0702 20:14:31.587664     448 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0702 20:14:50.976790     448 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


/kaggle/working/pgvector


I0702 20:14:52.375425     448 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0702 20:15:09.574803     448 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


/kaggle/working


In [30]:
import subprocess

# Arrancar el servicio
subprocess.run(["service", "postgresql", "start"], check=True)

# Ejecutar cada sentencia en su propio comando psql (evita el problema de transacción)
subprocess.run(["su", "-", "postgres", "-c", "psql -c \"ALTER USER postgres PASSWORD 'postgres';\""], check=True)
subprocess.run(["su", "-", "postgres", "-c", "psql -c \"DROP DATABASE IF EXISTS wikipedia_db;\""])
subprocess.run(["su", "-", "postgres", "-c", "psql -c \"CREATE DATABASE wikipedia_db;\""], check=True)

print("Usuario y base de datos configurados correctamente.")

 * Starting PostgreSQL 14 database server
   ...done.
ALTER ROLE
DROP DATABASE


NOTICE:  database "wikipedia_db" does not exist, skipping
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


CREATE DATABASE
Usuario y base de datos configurados correctamente.


In [31]:
!pip install -q psycopg2-binary

import psycopg2

conn = psycopg2.connect(
    host="localhost",
    dbname="wikipedia_db",
    user="postgres",
    password="postgres"
)
conn.autocommit = True
cur = conn.cursor()

# Habilitar la extensión pgvector en esta base de datos
cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
print("Extensión pgvector habilitada correctamente.")

I0702 20:17:07.331228     448 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
/usr/lib/python3.12/pty.py:95: DeprecationWarning: This process (pid=448) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


Extensión pgvector habilitada correctamente.


In [32]:
dim = embeddings.shape[1]

cur.execute(f"""
    DROP TABLE IF EXISTS documents;
    CREATE TABLE documents (
        id INTEGER PRIMARY KEY,
        text TEXT,
        embedding VECTOR({dim}),
        doc_id INTEGER,
        chunk_id INTEGER
    );
""")
print("Tabla 'documents' creada.")

Tabla 'documents' creada.


In [33]:
from psycopg2.extras import execute_values

print("Preparando datos para insertar...")
rows = [
    (
        int(idx),
        row["text"],
        embeddings[idx].tolist(),
        int(row["doc_id"]),
        int(row["chunk_id"])
    )
    for idx, row in chunks_df.iterrows()
]

insert_query = """
    INSERT INTO documents (id, text, embedding, doc_id, chunk_id)
    VALUES %s
"""

batch_size = 5000
print("Insertando en PostgreSQL...")
for i in range(0, len(rows), batch_size):
    batch = rows[i:i+batch_size]
    execute_values(cur, insert_query, batch, template="(%s, %s, %s::vector, %s, %s)")
    print(f"Procesado lote: {i} a {min(i+batch_size, len(rows))}")

cur.execute("SELECT COUNT(*) FROM documents;")
print(f"Total insertado: {cur.fetchone()[0]}")

Preparando datos para insertar...
Insertando en PostgreSQL...
Procesado lote: 0 a 5000
Procesado lote: 5000 a 10000
Procesado lote: 10000 a 15000
Procesado lote: 15000 a 20000
Procesado lote: 20000 a 25000
Procesado lote: 25000 a 30000
Procesado lote: 30000 a 35000
Procesado lote: 35000 a 40000
Procesado lote: 40000 a 45000
Procesado lote: 45000 a 50000
Procesado lote: 50000 a 55000
Procesado lote: 55000 a 60000
Procesado lote: 60000 a 65000
Procesado lote: 65000 a 70000
Procesado lote: 70000 a 75000
Procesado lote: 75000 a 79104
Total insertado: 79104


In [38]:
def pgvector_search(query_embedding, k=5):
    query_list = query_embedding[0].tolist() if query_embedding.ndim == 2 else query_embedding.tolist()
    query_str = str(query_list)

    cur.execute(f"""
        SELECT id, text, doc_id, chunk_id, embedding <=> %s::vector AS distance
        FROM documents
        ORDER BY distance ASC
        LIMIT %s;
    """, (query_str, k))

    results = []
    for row in cur.fetchall():
        id_, text, doc_id, chunk_id, distance = row
        metadata = {"doc_id": doc_id, "chunk_id": chunk_id}
        results.append((id_, distance, text, metadata))

    return results

print("=== Resultados pgvector (k=5) ===")
resultados = pgvector_search(query_vec, k=5)
for res in resultados:
    print(f"ID: {res[0]} | Distancia: {res[1]:.4f}")
    print(f"Metadata: {res[3]}")
    print(f"Texto: {res[2][:150]}...\n")

=== Resultados pgvector (k=5) ===
ID: 10176 | Distancia: 0.1297
Metadata: {'doc_id': 1391, 'chunk_id': 0}
Texto: Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going from a simple device for testing ...

ID: 1 | Distancia: 0.1382
Metadata: {'doc_id': 1, 'chunk_id': 0}
Texto: Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a battery. This will usually be a visu...

ID: 10177 | Distancia: 0.1599
Metadata: {'doc_id': 1391, 'chunk_id': 1}
Texto: ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries. Their common principle is ba...

ID: 37406 | Distancia: 0.1609
Metadata: {'doc_id': 5067, 'chunk_id': 1}
Texto: ils. One was connected via a series resistor to the battery supply. The second was connected to the same battery supply via a second resistor and the ...

ID: 71872 | Distancia: 0.1614
Metadat

**¿Qué tan "explicable" te parece esta aproximación vs las otras?**  
Es la más explícita y transparente de todas: la consulta de similitud es SQL puro
(`ORDER BY embedding <=> query_vector LIMIT k`), sin abstracciones de cliente
propietario. Se puede inspeccionar, depurar y optimizar con las mismas herramientas
que cualquier consulta SQL, lo cual la hace más fácil de auditar.

**¿Qué ventajas ofrece el mundo SQL (JOIN, filtros, agregaciones)?**  
Al vivir la tabla `documents` dentro de un RDBMS normal, se pueden combinar
búsquedas por similitud con JOINs a otras tablas (por ejemplo, unir con una tabla
de autores o categorías), agregaciones (COUNT, AVG por doc_id), filtros WHERE complejos, y todo dentro de transacciones ACID — algo que las bases vectoriales dedicadas normalmente no ofrecen con la misma madurez.

**¿Qué limitaciones esperas en escalabilidad frente a bases vectoriales dedicadas?**  
Sin un índice vectorial, la consulta hace un escaneo secuencial completo de la tabla, lo cual escala mal a medida que crece el número de filas. Incluso con índices ANN, pgvector tiende a quedarse por detrás de motores dedicados (Qdrant, Milvus) en throughput y latencia a gran escala, ya que estos están optimizados específicamente para operaciones vectoriales masivas, mientras que Postgres es un motor de propósito general adaptado para soportar vectores.